# YuNet FaceDetectorYN 클래스 분석 및 인덱스 에러 해결

이 노트북에서는 FaceDetectorYN 클래스의 구조를 분석하고, 발생한 인덱스 에러의 원인을 찾아 수정하는 과정을 진행합니다.

## 1. 필요 라이브러리 임포트

In [1]:
import cv2 as cv
import numpy as np
import onnxruntime as ort
import os
import sys

# 현재 프로젝트 경로 추가
sys.path.append('/home/aa/hongOpencv/python')

print(f"OpenCV version: {cv.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"ONNXRuntime version: {ort.__version__}")

OpenCV version: 4.12.0
NumPy version: 2.2.6
ONNXRuntime version: 1.22.1


## 2. FaceDetectorYN 클래스 구조 분석

OpenCV의 FaceDetectorYN 클래스와 사용자 정의 YuNet 클래스를 비교하여 구조를 분석합니다.

In [ ]:
# OpenCV FaceDetectorYN 클래스 테스트
model_path = "/home/aa/hongOpencv/data/face_detection_yunet_2023mar.onnx"
input_size = (320, 320)

try:
    # OpenCV FaceDetectorYN 생성
    face_detector = cv.FaceDetectorYN.create(
        model=model_path,
        config="",
        input_size=input_size,
        score_threshold=0.6,
        nms_threshold=0.3,
        top_k=5000
    )
    print("OpenCV FaceDetectorYN 생성 성공")
    print(f"Input size: {input_size}")

    # 테스트 이미지 로드
    test_image = cv.imread("/home/aa/hongOpencv/data/lenna.bmp")
    if test_image is not None:
        print(f"테스트 이미지 shape: {test_image.shape}")

        # 얼굴 검출 수행
        faces = face_detector.detect(test_image)
        print(f"검출 결과 타입: {type(faces)}")
        print(f"검출 결과 길이: {len(faces) if faces else 0}")
        if faces[1] is not None:
            print(f"faces[1] shape: {faces[1].shape}")
            print(f"검출된 얼굴 수: {faces[1].shape[0]}")
        else:
            print("검출된 얼굴이 없습니다.")
    else:
        print("테스트 이미지를 로드할 수 없습니다.")

except Exception as e:
    print(f"OpenCV FaceDetectorYN 오류: {e}")

## 3. ONNX 모델 출력 구조 직접 분석

ONNX 모델을 직접 로드하여 출력 구조를 확인하고 인덱스 에러의 원인을 찾습니다.

In [2]:
# ONNX 모델 직접 로드 및 출력 구조 분석
model_path = "/home/aa/hongOpencv/data/face_detection_yunet_2023mar.onnx"

try:
    # ONNX Runtime 세션 생성
    sess = ort.InferenceSession(model_path)

    # 입력 정보 확인
    input_info = sess.get_inputs()[0]
    print(f"입력 이름: {input_info.name}")
    print(f"입력 shape: {input_info.shape}")
    print(f"입력 타입: {input_info.type}")

    print("\n=== 출력 정보 ===")
    for i, output in enumerate(sess.get_outputs()):
        print(f"{i}: {output.name} - shape: {output.shape}, type: {output.type}")

    # 테스트 입력 생성 (더미 데이터)
    input_shape = [1, 3, 640, 640]  # ONNX 모델의 입력 크기
    test_input = np.random.random(input_shape).astype(np.float32)

    # 추론 실행
    output_names = [o.name for o in sess.get_outputs()]
    results = sess.run(output_names, {input_info.name: test_input})

    print(f"\n=== 실제 출력 shape 확인 ===")
    for i, (name, result) in enumerate(zip(output_names, results)):
        print(f"{i}: {name} -> shape: {result.shape}, dtype: {result.dtype}")

    # cls 출력들 상세 분석
    cls_outputs = [r for n, r in zip(output_names, results) if n.startswith('cls_')]
    print(f"\n=== Classification 출력 분석 ===")
    for i, cls_out in enumerate(cls_outputs):
        print(f"cls_{i}: shape={cls_out.shape}, last_dim_size={cls_out.shape[-1]}")
        if cls_out.shape[-1] > 1:
            print(f"  - 다중 클래스 분류기 (배경/얼굴)")
        else:
            print(f"  - 단일 클래스 분류기 (얼굴 존재 여부)")

except Exception as e:
    print(f"ONNX 모델 분석 중 오류: {e}")

입력 이름: input
입력 shape: [1, 3, 640, 640]
입력 타입: tensor(float)

=== 출력 정보 ===
0: cls_8 - shape: [1, 6400, 1], type: tensor(float)
1: cls_16 - shape: [1, 1600, 1], type: tensor(float)
2: cls_32 - shape: [1, 400, 1], type: tensor(float)
3: obj_8 - shape: [1, 6400, 1], type: tensor(float)
4: obj_16 - shape: [1, 1600, 1], type: tensor(float)
5: obj_32 - shape: [1, 400, 1], type: tensor(float)
6: bbox_8 - shape: [1, 6400, 4], type: tensor(float)
7: bbox_16 - shape: [1, 1600, 4], type: tensor(float)
8: bbox_32 - shape: [1, 400, 4], type: tensor(float)
9: kps_8 - shape: [1, 6400, 10], type: tensor(float)
10: kps_16 - shape: [1, 1600, 10], type: tensor(float)
11: kps_32 - shape: [1, 400, 10], type: tensor(float)

=== 실제 출력 shape 확인 ===
0: cls_8 -> shape: (1, 6400, 1), dtype: float32
1: cls_16 -> shape: (1, 1600, 1), dtype: float32
2: cls_32 -> shape: (1, 400, 1), dtype: float32
3: obj_8 -> shape: (1, 6400, 1), dtype: float32
4: obj_16 -> shape: (1, 1600, 1), dtype: float32
5: obj_32 -> shape: (1

## 4. 인덱스 에러 원인 분석

에러 메시지 `IndexError: index 1 is out of bounds for axis 2 with size 1`을 기반으로 원인을 분석합니다.

이 에러는 `cls[..., 1]`에서 발생하는데, cls의 마지막 차원이 크기 1이므로 인덱스 1에 접근할 수 없다는 의미입니다.

In [3]:
# 인덱스 에러 재현 및 원인 확인
print("=== 인덱스 에러 재현 ===")

# 에러가 발생한 상황 시뮬레이션
# cls의 shape이 [1, N, 1]인 경우 (단일 클래스)
cls_single = np.random.random((1, 100, 1))  # [1, N, 1]
obj = np.random.random((1, 100, 1))          # [1, N, 1]

print(f"cls shape: {cls_single.shape}")
print(f"cls 마지막 차원 크기: {cls_single.shape[-1]}")

try:
    # 기존 코드 (에러 발생)
    scores_error = cls_single[..., 1] * obj[..., 0]  # 인덱스 1 접근 시도
    print("에러가 발생하지 않았습니다.")
except IndexError as e:
    print(f"예상된 IndexError: {e}")

# 올바른 방법 (단일 클래스인 경우)
scores_correct = cls_single[..., 0] * obj[..., 0]
print(f"올바른 접근 결과 shape: {scores_correct.shape}")

print("\n=== 다중 클래스인 경우 비교 ===")
# cls의 shape이 [1, N, 2]인 경우 (배경/얼굴)
cls_multi = np.random.random((1, 100, 2))  # [1, N, 2]

print(f"다중 클래스 cls shape: {cls_multi.shape}")
print(f"다중 클래스 마지막 차원 크기: {cls_multi.shape[-1]}")

# 다중 클래스에서는 인덱스 1 접근 가능
scores_multi = cls_multi[..., 1] * obj[..., 0]  # 얼굴 클래스
print(f"다중 클래스 결과 shape: {scores_multi.shape}")

=== 인덱스 에러 재현 ===
cls shape: (1, 100, 1)
cls 마지막 차원 크기: 1
예상된 IndexError: index 1 is out of bounds for axis 2 with size 1
올바른 접근 결과 shape: (1, 100)

=== 다중 클래스인 경우 비교 ===
다중 클래스 cls shape: (1, 100, 2)
다중 클래스 마지막 차원 크기: 2
다중 클래스 결과 shape: (1, 100)


## 5. 수정된 _postprocess 메서드 구현

인덱스 에러를 해결하기 위한 수정된 코드를 구현합니다.

In [ ]:
def fixed_postprocess_scores(cls, obj):
    """
    cls와 obj 텐서로부터 올바른 점수 계산

    Args:
        cls: Classification 출력 [1, N, C] (C=1 또는 2)
        obj: Objectness 출력 [1, N, 1]

    Returns:
        scores: [N] shape의 점수 배열
    """
    # Sigmoid 활성화 적용
    cls_prob = 1 / (1 + np.exp(-cls))
    obj_prob = 1 / (1 + np.exp(-obj))

    # 클래스 차원 확인 후 적절한 인덱스 사용
    if cls_prob.shape[-1] == 1:
        # 단일 클래스: 얼굴 존재 여부만
        face_prob = cls_prob[..., 0]
        print("단일 클래스 분류기 사용: cls[..., 0]")
    elif cls_prob.shape[-1] == 2:
        # 다중 클래스: [배경, 얼굴]
        face_prob = cls_prob[..., 1]  # 얼굴 클래스
        print("다중 클래스 분류기 사용: cls[..., 1]")
    else:
        raise ValueError(f"예상치 못한 클래스 차원: {cls_prob.shape[-1]}")

    # 최종 점수: 얼굴 확률 × 객체 존재 확률
    scores = (face_prob * obj_prob[..., 0]).squeeze(0)
    return scores

# 테스트
print("=== 수정된 함수 테스트 ===")

# 단일 클래스 테스트
cls_single = np.random.random((1, 5, 1)) * 2 - 1  # logit 값
obj_single = np.random.random((1, 5, 1)) * 2 - 1
scores_single = fixed_postprocess_scores(cls_single, obj_single)
print(f"단일 클래스 결과 shape: {scores_single.shape}")
print(f"단일 클래스 점수 범위: [{scores_single.min():.3f}, {scores_single.max():.3f}]")

print()

# 다중 클래스 테스트
cls_multi = np.random.random((1, 5, 2)) * 2 - 1   # logit 값
obj_multi = np.random.random((1, 5, 1)) * 2 - 1
scores_multi = fixed_postprocess_scores(cls_multi, obj_multi)
print(f"다중 클래스 결과 shape: {scores_multi.shape}")
print(f"다중 클래스 점수 범위: [{scores_multi.min():.3f}, {scores_multi.max():.3f}]")

## 6. yunet_ort.py 파일 수정사항 요약

다음과 같이 `_postprocess` 메서드의 점수 계산 부분을 수정해야 합니다:

```python
# 기존 코드 (에러 발생)
scores = (cls[..., 1] * obj[..., 0]).squeeze(0)

# 수정된 코드
if cls.shape[-1] == 1:
    # 단일 클래스 분류기: 얼굴 존재 여부만
    scores = (cls[..., 0] * obj[..., 0]).squeeze(0)
else:
    # 다중 클래스 분류기: [배경, 얼굴]
    scores = (cls[..., 1] * obj[..., 0]).squeeze(0)
```

이 수정으로 인덱스 에러를 해결하고 단일/다중 클래스 모델 모두를 지원할 수 있습니다.

In [ ]:
# 실제 yunet_ort.py 파일의 해당 부분을 직접 확인해보겠습니다
import os

yunet_ort_path = "/home/aa/hongOpencv/python/yunet_ort.py"

if os.path.exists(yunet_ort_path):
    with open(yunet_ort_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    # 문제가 되는 라인을 찾습니다 (scores = ... 라인)
    for i, line in enumerate(lines):
        if 'scores = (cls[..., 1] * obj[..., 0]).squeeze(0)' in line:
            print(f"문제 라인 발견 (라인 {i+1}): {line.strip()}")
            print(f"이전 라인 {i}: {lines[i-1].strip() if i > 0 else 'N/A'}")
            print(f"다음 라인 {i+2}: {lines[i+1].strip() if i < len(lines)-1 else 'N/A'}")
            break
    else:
        print("해당 라인을 찾을 수 없습니다.")

        # 대신 cls와 관련된 라인들을 찾아보겠습니다
        print("\n=== cls 관련 라인들 ===")
        for i, line in enumerate(lines):
            if 'cls' in line and ('scores' in line or '[...' in line):
                print(f"라인 {i+1}: {line.strip()}")
else:
    print(f"파일을 찾을 수 없습니다: {yunet_ort_path}")

print("\n수정이 필요한 코드:")
print("기존: scores = (cls[..., 1] * obj[..., 0]).squeeze(0)")
print("수정: ")
print("if cls.shape[-1] == 1:")
print("    scores = (cls[..., 0] * obj[..., 0]).squeeze(0)")
print("else:")
print("    scores = (cls[..., 1] * obj[..., 0]).squeeze(0)")

In [4]:
# Prior 계산 vs ONNX 출력 크기 비교
print("=== Prior vs ONNX 출력 크기 분석 ===")

# ONNX 출력에서 실제 anchor 수 계산
onnx_anchor_counts = [6400, 1600, 400]  # cls_8, cls_16, cls_32
total_onnx_anchors = sum(onnx_anchor_counts)
print(f"ONNX 총 anchor 수: {total_onnx_anchors}")

# YuNet prior 계산 (640x640 기준)
input_w, input_h = 640, 640
steps = [8, 16, 32, 64]  # 원래 코드의 설정
min_sizes = [[10, 16, 24], [32, 48], [64, 96], [128, 192, 256]]

def calculate_priors(in_w, in_h, steps, min_sizes):
    feature_maps = []
    for s in steps:
        feature_maps.append((int(np.ceil(in_h / s)), int(np.ceil(in_w / s))))

    total_priors = 0
    for k, (fm_h, fm_w) in enumerate(feature_maps):
        stride_priors = fm_h * fm_w * len(min_sizes[k])
        print(f"Stride {steps[k]}: {fm_h}x{fm_w} x {len(min_sizes[k])} = {stride_priors}")
        total_priors += stride_priors

    return total_priors, feature_maps

original_total, feature_maps = calculate_priors(640, 640, steps, min_sizes)
print(f"\n원래 설정으로 계산된 총 prior 수: {original_total}")
print(f"ONNX 출력과의 차이: {original_total - total_onnx_anchors}")

# ONNX 출력에 맞는 설정 찾기
print(f"\n=== ONNX 출력에 맞는 설정 찾기 ===")
print("ONNX 출력 분석:")
for i, count in enumerate([6400, 1600, 400]):
    stride = [8, 16, 32][i]
    fm_size = int(np.sqrt(count / 3))  # 각 위치에 3개 anchor 가정
    print(f"  {stride}: {count} anchors -> {fm_size}x{fm_size} x 3")

# 올바른 설정 추정
corrected_steps = [8, 16, 32]  # stride 64 제거
corrected_min_sizes = [[10, 16, 24], [32, 48], [64, 96]]  # 마지막 제거

corrected_total, _ = calculate_priors(640, 640, corrected_steps, corrected_min_sizes)
print(f"\n수정된 설정으로 계산된 총 prior 수: {corrected_total}")
print(f"ONNX 출력과의 일치: {corrected_total == total_onnx_anchors}")

=== Prior vs ONNX 출력 크기 분석 ===
ONNX 총 anchor 수: 8400
Stride 8: 80x80 x 3 = 19200
Stride 16: 40x40 x 2 = 3200
Stride 32: 20x20 x 2 = 800
Stride 64: 10x10 x 3 = 300

원래 설정으로 계산된 총 prior 수: 23500
ONNX 출력과의 차이: 15100

=== ONNX 출력에 맞는 설정 찾기 ===
ONNX 출력 분석:
  8: 6400 anchors -> 46x46 x 3
  16: 1600 anchors -> 23x23 x 3
  32: 400 anchors -> 11x11 x 3
Stride 8: 80x80 x 3 = 19200
Stride 16: 40x40 x 2 = 3200
Stride 32: 20x20 x 2 = 800

수정된 설정으로 계산된 총 prior 수: 23200
ONNX 출력과의 일치: False


In [5]:
# 더 정확한 분석: ONNX 출력으로부터 역산
print("=== 정확한 설정 역산 ===")

# ONNX 출력: 6400, 1600, 400
# 640x640 입력 기준으로 각 stride별 feature map 크기 계산

def reverse_calculate_settings(input_size, anchor_counts, strides):
    """ONNX 출력으로부터 설정 역산"""
    print(f"입력 크기: {input_size}x{input_size}")

    calculated_settings = []
    for stride, count in zip(strides, anchor_counts):
        fm_size = input_size // stride
        expected_count = fm_size * fm_size

        if count % expected_count == 0:
            anchors_per_location = count // expected_count
            print(f"Stride {stride}: {fm_size}x{fm_size} = {expected_count} locations")
            print(f"  -> {count} outputs = {anchors_per_location} anchors per location")
            calculated_settings.append((stride, anchors_per_location))
        else:
            print(f"Stride {stride}: {count} outputs don't match {fm_size}x{fm_size} grid")

    return calculated_settings

# 640x640 모델 분석
settings_640 = reverse_calculate_settings(640, [6400, 1600, 400], [8, 16, 32])
print(f"\\n계산된 설정: {settings_640}")

# 총 anchor 수 검증
total_calculated = sum([6400, 1600, 400])
print(f"총 계산된 anchor 수: {total_calculated}")

# 이제 올바른 prior 생성을 위한 설정 도출
print("\\n=== 올바른 YuNet 설정 ===")
correct_steps = [8, 16, 32]
# 각 stride별 anchor 개수에 맞춰 min_sizes 조정
correct_min_sizes = [
    [10, 16, 24],      # stride 8: 3개 anchor
    [32, 48],          # stride 16: 2개 anchor
    [64, 96]           # stride 32: 2개 anchor (400/(20*20) = 1이므로 실제로는 1개?)
]

# 다시 계산해보기
print("stride 32 재검토:")
fm32 = 640 // 32
print(f"stride 32 feature map: {fm32}x{fm32} = {fm32*fm32}")
print(f"ONNX 출력: 400")
print(f"400 / {fm32*fm32} = {400/(fm32*fm32)} anchors per location")

=== 정확한 설정 역산 ===
입력 크기: 640x640
Stride 8: 80x80 = 6400 locations
  -> 6400 outputs = 1 anchors per location
Stride 16: 40x40 = 1600 locations
  -> 1600 outputs = 1 anchors per location
Stride 32: 20x20 = 400 locations
  -> 400 outputs = 1 anchors per location
\n계산된 설정: [(8, 1), (16, 1), (32, 1)]
총 계산된 anchor 수: 8400
\n=== 올바른 YuNet 설정 ===
stride 32 재검토:
stride 32 feature map: 20x20 = 400
ONNX 출력: 400
400 / 400 = 1.0 anchors per location
